# Weekend Churn Campaign Demo
This notebook simulates weekend-focused campaign data and performs a simple ETL flow for dashboard-style reporting.

It is intentionally self-contained:
- synthetic data is generated inside the notebook
- data is cleaned and standardized with Spark
- a joined, analytics-ready table is created
- summary metrics are exposed as a temp view for BI / SQL exploration


In [ ]:
from datetime import date
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import random

spark.conf.set("spark.sql.shuffle.partitions", "4")

random.seed(42)

## 1) Simulate weekend campaign and response data

In [ ]:
# Generate a small synthetic data set with weekend campaigns (Fri/Sat/Sun)
weekend_dates = [
    date(2026, 4, 3),   # Friday
    date(2026, 4, 4),   # Saturday
    date(2026, 4, 5),   # Sunday
    date(2026, 4, 10),
    date(2026, 4, 11),
    date(2026, 4, 12),
]

channels = ["email", "sms", "push", "paid_social"]
segments = ["new", "active", "at_risk", "lapsed"]
regions = ["NA", "EMEA", "APAC"]

rows = []

for d in weekend_dates:
    day_name = d.strftime("%a")
    for channel in channels:
        for segment in segments:
            for region in regions:
                base = {
                    "email": 120,
                    "sms": 80,
                    "push": 95,
                    "paid_social": 150
                }[channel]

                # Weekend effect:
                # - Saturday tends to have strong engagement
                # - Sunday slightly better conversion on reactivation
                if day_name == "Sat":
                    multiplier = 1.15
                elif day_name == "Sun":
                    multiplier = 1.08
                else:
                    multiplier = 0.92

                seg_multiplier = {
                    "new": 1.10,
                    "active": 1.00,
                    "at_risk": 0.95,
                    "lapsed": 0.85
                }[segment]

                customers_reached = int(base * multiplier * seg_multiplier)

                open_rate = {
                    "email": 0.33,
                    "sms": 0.12,
                    "push": 0.18,
                    "paid_social": 0.06
                }[channel]

                click_rate = {
                    "email": 0.11,
                    "sms": 0.05,
                    "push": 0.07,
                    "paid_social": 0.03
                }[channel]

                if day_name == "Sat":
                    open_rate += 0.04
                    click_rate += 0.015
                elif day_name == "Sun":
                    open_rate += 0.025
                    click_rate += 0.010

                opens = int(customers_reached * open_rate)
                clicks = int(customers_reached * click_rate)

                conversions = max(0, int(clicks * {
                    "email": 0.24,
                    "sms": 0.30,
                    "push": 0.20,
                    "paid_social": 0.10
                }[channel]))

                churned = max(0, int(customers_reached * {
                    "new": 0.018,
                    "active": 0.042,
                    "at_risk": 0.078,
                    "lapsed": 0.092
                }[segment]))

                churn_adjustment = 0.92 if day_name in ("Sat", "Sun") and segment in ("at_risk", "lapsed") else 1.0
                churned = int(churned * churn_adjustment)

                rows.append((
                    f"WKND-{d.strftime('%Y%m%d')}-{channel[:2].upper()}-{segment[:2].upper()}-{region}",
                    d.isoformat(),
                    day_name,
                    channel,
                    segment,
                    region,
                    customers_reached,
                    opens,
                    clicks,
                    conversions,
                    churned
                ))

schema = StructType([
    StructField("campaign_id", StringType(), False),
    StructField("campaign_date", StringType(), False),
    StructField("day_name", StringType(), False),
    StructField("channel", StringType(), False),
    StructField("segment", StringType(), False),
    StructField("region", StringType(), False),
    StructField("customers_reached", IntegerType(), False),
    StructField("opens", IntegerType(), False),
    StructField("clicks", IntegerType(), False),
    StructField("conversions", IntegerType(), False),
    StructField("churned_customers", IntegerType(), False),
])

campaign_raw = spark.createDataFrame(rows, schema=schema)
campaign_raw.show(10, truncate=False)

## 2) Clean and standardize the data

In [ ]:
campaign_clean = (
    campaign_raw
    .withColumn("campaign_date", F.to_date("campaign_date"))
    .withColumn("is_weekend", F.lit(True))
    .withColumn("open_rate", F.round(F.col("opens") / F.col("customers_reached"), 4))
    .withColumn("click_rate", F.round(F.col("clicks") / F.col("customers_reached"), 4))
    .withColumn("conversion_rate", F.round(F.col("conversions") / F.col("customers_reached"), 4))
    .withColumn("churn_rate", F.round(F.col("churned_customers") / F.col("customers_reached"), 4))
    .withColumn("engagement_score", F.round((F.col("opens") * 0.4 + F.col("clicks") * 0.6) / F.col("customers_reached"), 4))
    .withColumn(
        "campaign_theme",
        F.when(F.col("segment").isin("at_risk", "lapsed"), F.lit("retention"))
         .when(F.col("segment") == "new", F.lit("onboarding"))
         .otherwise(F.lit("growth"))
    )
)

campaign_clean.printSchema()
campaign_clean.show(10, truncate=False)

## 3) Build dashboard-ready aggregates

In [ ]:
dashboard_ready = (
    campaign_clean
    .groupBy("campaign_date", "day_name", "channel", "segment", "region", "campaign_theme")
    .agg(
        F.sum("customers_reached").alias("customers_reached"),
        F.sum("opens").alias("opens"),
        F.sum("clicks").alias("clicks"),
        F.sum("conversions").alias("conversions"),
        F.sum("churned_customers").alias("churned_customers"),
        F.round(F.avg("open_rate"), 4).alias("avg_open_rate"),
        F.round(F.avg("click_rate"), 4).alias("avg_click_rate"),
        F.round(F.avg("conversion_rate"), 4).alias("avg_conversion_rate"),
        F.round(F.avg("churn_rate"), 4).alias("avg_churn_rate"),
        F.round(F.avg("engagement_score"), 4).alias("avg_engagement_score")
    )
    .withColumn("roi_proxy", F.round((F.col("conversions") * 12.0 - F.col("customers_reached") * 1.5), 2))
    .withColumn("retention_proxy", F.round(1.0 - F.col("avg_churn_rate"), 4))
)

dashboard_ready.orderBy("campaign_date", "channel", "segment", "region").show(20, truncate=False)

## 4) Register a temp view for SQL / BI

In [ ]:
dashboard_ready.createOrReplaceTempView("vw_weekend_churn_campaign_dashboard")

spark.sql('''
SELECT
    campaign_date,
    day_name,
    channel,
    segment,
    region,
    customers_reached,
    avg_open_rate,
    avg_click_rate,
    avg_churn_rate,
    roi_proxy
FROM vw_weekend_churn_campaign_dashboard
ORDER BY campaign_date, channel, segment, region
''').show(20, truncate=False)

## 5) Example dashboard cuts

In [ ]:
# Weekend lift by channel
spark.sql('''
SELECT
    channel,
    ROUND(AVG(avg_open_rate), 4) AS mean_open_rate,
    ROUND(AVG(avg_click_rate), 4) AS mean_click_rate,
    ROUND(AVG(avg_churn_rate), 4) AS mean_churn_rate,
    ROUND(AVG(retention_proxy), 4) AS mean_retention_proxy
FROM vw_weekend_churn_campaign_dashboard
GROUP BY channel
ORDER BY mean_retention_proxy DESC
''').show(truncate=False)

# Top retention opportunities
spark.sql('''
SELECT
    campaign_date,
    day_name,
    channel,
    segment,
    region,
    customers_reached,
    avg_churn_rate,
    retention_proxy,
    roi_proxy
FROM vw_weekend_churn_campaign_dashboard
ORDER BY avg_churn_rate DESC, roi_proxy ASC
LIMIT 10
''').show(truncate=False)

## 6) Optional: persist a curated output table

In a real AI Data Platform workflow, this would typically be written to a governed Delta/Parquet target for downstream dashboarding.


In [ ]:
# Example write path (commented out for demo portability)
# dashboard_ready.write.mode("overwrite").format("parquet").save("/tmp/weekend_churn_campaign_dashboard")

print("Notebook demo complete.")